# Deep Learning-Based Multi-Class Lung Disease Detection from Chest X-Ray Images Using Transfer Learning and Explainable AI

This notebook is a complete Colab-ready pipeline for the thesis workflow described in the project instructions.

Classes: Normal, Pneumonia, Tuberculosis


## 01. Introduction & Configuration

This thesis develops a deep learning pipeline for multiclass chest X-ray classification. The goal is to classify images into Normal, Pneumonia, and Tuberculosis using transfer learning and explainable AI. All experiments use one consistent dataset split, preprocessing pipeline, and evaluation strategy so the results are comparable across models.

The notebook follows a complete research workflow: data verification, cleaning, splitting, preprocessing, training, testing, comparison, and Grad-CAM explainability.

In [ ]:
# 01. Introduction & Configuration

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 3

CLASSES = ["Normal", "Pneumonia", "Tuberculosis"]
LABEL_MAP = {"Normal": 0, "Pneumonia": 1, "Tuberculosis": 2}
REFERENCE_COUNTS = {"Normal": 6074, "Pneumonia": 4530, "Tuberculosis": 1054}

USE_CLASS_WEIGHTS = True
USE_OVERSAMPLING = False
USE_IMAGE_LEVEL_SPLIT = True
TARGET_CLASSES = ["Normal", "Pneumonia", "Tuberculosis"]

CONFIG = {
    'title': 'Deep Learning-Based Multi-Class Lung Disease Detection from Chest X-Ray Images Using Transfer Learning and Explainable AI',
    'seed': SEED,
    'image_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'num_classes': NUM_CLASSES,
    'classes': CLASSES,
    'label_map': LABEL_MAP,
    'reference_counts': REFERENCE_COUNTS,
    'use_class_weights': USE_CLASS_WEIGHTS,
    'use_oversampling': USE_OVERSAMPLING,
    'use_image_level_split': USE_IMAGE_LEVEL_SPLIT,
    'data_leakage_prevention': 'The unified dataset does not provide a reliable cross-source subject identifier, so a stratified image-level split (70/15/15) was used to preserve class balance while avoiding false subject-level assumptions.'
}

print('Thesis title: Deep Learning-Based Multi-Class Lung Disease Detection from Chest X-Ray Images Using Transfer Learning and Explainable AI')
print('Target classes:', CLASSES)
print('Label mapping:', LABEL_MAP)
print('Reference dataset counts:', REFERENCE_COUNTS)
print('No COPD class included in this thesis experiment.')
print('Reproducibility config ready.')

## 02. Import Libraries

The next cells import the libraries for data handling, plotting, PyTorch model building, and evaluation. These packages are required for dataset preprocessing, model training, and metric reporting in a Colab environment.

Key libraries include:
- Python: os, random, numpy, pandas
- Visualization: matplotlib, seaborn
- Medical image handling: PIL
- Deep learning: PyTorch, torchvision
- Evaluation: scikit-learn

In [ ]:
# 02. Import Libraries
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

print("Libraries imported successfully.")

## 03. Google Drive Setup

This section mounts Google Drive and creates the folder structure used for the dataset, splits, models, and thesis results. The workflow assumes your dataset is stored in a Drive folder like: `/content/drive/MyDrive/Thesis/Dataset`.

Using Drive makes the project reproducible and lets the model checkpoints and result files be saved outside the runtime session.

In [ ]:
# 03. Google Drive Setup
from google.colab import drive

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Thesis'
DATASET_DIR = os.path.join(BASE_DIR, 'Dataset')
MODEL_DIR = os.path.join(BASE_DIR, 'Models')
RESULT_DIR = os.path.join(BASE_DIR, 'Results')
SPLIT_DIR = os.path.join(BASE_DIR, 'Splits')

for path in [BASE_DIR, DATASET_DIR, MODEL_DIR, RESULT_DIR, SPLIT_DIR]:
    os.makedirs(path, exist_ok=True)

print('Drive mounted and project folders created.')

## 04. Dataset Verification


In [ ]:
# 05. Dataset Statistics
classes = CLASSES
label_map = LABEL_MAP

for class_name in classes:
    class_path = os.path.join(DATASET_DIR, class_name)
    if not os.path.exists(class_path):
        print(f'Warning: folder not found -> {class_path}')
        continue

    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ]

    print(f'{class_name}: {len(images)} images')

print('Dataset includes exactly the three thesis classes: Normal, Pneumonia, Tuberculosis.')
print('The code preserves source metadata for fair split and analysis without creating any synthetic subject identifiers.')

## 05.1 Dataset Provenance

This table summarizes the final unified dataset by source. The counts are computed directly from the merged labels file and therefore reflect the actual dataset composition used in the thesis. This provenance record is essential because the dataset combines multiple public and institutional sources for Normal, Pneumonia, and Tuberculosis cases.

In [ ]:
# 05.1 Dataset Provenance
provenance_path = os.path.join(BASE_DIR, 'UnifiedDataset', 'labels.csv')
if not os.path.exists(provenance_path):
    provenance_path = os.path.join(os.getcwd(), 'UnifiedDataset', 'labels.csv')

if os.path.exists(provenance_path):
    provenance_df = pd.read_csv(provenance_path)

    if {'source_dataset', 'label'}.issubset(provenance_df.columns):
        provenance_table = (
            provenance_df
            .pivot_table(index='source_dataset', columns='label', aggfunc='size', fill_value=0)
            .reindex(columns=['Normal', 'Pneumonia', 'Tuberculosis'])
            .astype(int)
            .reset_index()
        )
        provenance_table.columns = ['Dataset Source', 'Normal', 'Pneumonia', 'Tuberculosis']
        provenance_table['Total'] = provenance_table[['Normal', 'Pneumonia', 'Tuberculosis']].sum(axis=1)

        total_row = pd.DataFrame({
            'Dataset Source': ['Total'],
            'Normal': [provenance_table['Normal'].sum()],
            'Pneumonia': [provenance_table['Pneumonia'].sum()],
            'Tuberculosis': [provenance_table['Tuberculosis'].sum()],
            'Total': [provenance_table['Total'].sum()],
        })

        provenance_table = pd.concat([provenance_table, total_row], ignore_index=True)
        print('Dataset provenance table (computed from final unified labels):')
        print(provenance_table.to_string(index=False))

        provenance_csv_path = os.path.join(RESULT_DIR, 'dataset_provenance.csv')
        provenance_table.to_csv(provenance_csv_path, index=False)
        print(f'Saved dataset provenance table to: {provenance_csv_path}')
    else:
        print('Warning: source_dataset/label columns not found in labels.csv; provenance table could not be generated.')
else:
    print('Warning: UnifiedDataset/labels.csv not found; provenance table could not be generated.')

## 05.2 Duplicate and Near-Duplicate Audit

Because the thesis dataset is unified from multiple sources, duplicate and near-duplicate chest X-rays can create train/test leakage if they appear in more than one source dataset. This section documents whether exact duplicate or visually similar images were checked before model training and splitting.

The method uses exact SHA-256 comparison for duplicates and a dHash perceptual-hash comparison for near-duplicate detection. This is a stronger methodology than relying on dataset labels alone.

In [ ]:
# 05.2 Duplicate and Near-Duplicate Audit
import hashlib


def sha256_file(path):
    hash_obj = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            hash_obj.update(chunk)
    return hash_obj.hexdigest()


def dhash_image(path, hash_size=8):
    with Image.open(path) as img:
        img = img.convert('L').resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
        pixels = np.asarray(img, dtype=np.uint8)
    diff = pixels[:, 1:] > pixels[:, :-1]
    return ''.join('1' if bit else '0' for bit in diff.flatten())


duplicate_records = []
for class_name in CLASSES:
    class_path = os.path.join(DATASET_DIR, class_name)
    if not os.path.exists(class_path):
        continue

    for filename in sorted(os.listdir(class_path)):
        if not filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            continue

        image_path = os.path.join(class_path, filename)
        duplicate_records.append({
            'class_name': class_name,
            'image_path': image_path,
            'file_hash': sha256_file(image_path),
            'phash': dhash_image(image_path),
        })

if duplicate_records:
    duplicate_df = pd.DataFrame(duplicate_records)

    exact_duplicate_mask = duplicate_df.duplicated(subset='file_hash', keep=False)
    near_duplicate_mask = duplicate_df.duplicated(subset='phash', keep=False)

    exact_duplicate_groups = int(duplicate_df[exact_duplicate_mask].groupby('file_hash').ngroups)
    exact_duplicate_instances = int(exact_duplicate_mask.sum())
    near_duplicate_groups = int(duplicate_df[near_duplicate_mask].groupby('phash').ngroups)
    near_duplicate_instances = int(near_duplicate_mask.sum())

    print(f'Exact duplicate image groups: {exact_duplicate_groups}')
    print(f'Exact duplicate image instances: {exact_duplicate_instances}')
    if exact_duplicate_mask.any():
        print(duplicate_df[exact_duplicate_mask][['class_name', 'image_path', 'file_hash']].head(10).to_string(index=False))

    print(f'Near-duplicate perceptual-hash groups: {near_duplicate_groups}')
    print(f'Near-duplicate image instances: {near_duplicate_instances}')
    print('Duplicate checking performed using exact SHA-256 hashing and dHash perceptual hashing to audit cross-dataset leakage risk before train/validation/test splitting.')
else:
    print('No image files were discovered in the dataset folder, so duplicate checks were not run.')

## 06. Corrupted Image Detection


In [ ]:
def check_corrupted_images(dataset_dir, class_names):
    corrupted = []

    for class_name in class_names:
        class_path = os.path.join(dataset_dir, class_name)
        if not os.path.exists(class_path):
            continue

        for filename in os.listdir(class_path):
            filepath = os.path.join(class_path, filename)

            if not filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                continue

            try:
                with Image.open(filepath) as img:
                    img.verify()
            except Exception:
                corrupted.append(filepath)

    return corrupted

corrupted_images = check_corrupted_images(DATASET_DIR, classes)
print('Corrupted images:', len(corrupted_images))
if corrupted_images:
    print(corrupted_images[:10])

## 07. Dataset Visualization


In [ ]:
# 07. Dataset Visualization
# Use the unified dataset metadata file as the source of truth for provenance.
# This keeps source_dataset, original_filename, and original_relative_path intact
# without inventing a synthetic subject identifier that is not available across the unified sources.

labels_path = os.path.join(BASE_DIR, 'UnifiedDataset', 'labels.csv')

if os.path.exists(labels_path):
    df = pd.read_csv(labels_path)
    df = df.rename(columns={
        'label': 'class_name',
        'filename': 'image_id',
        'relative_path': 'original_relative_path'
    })
    df['image_path'] = df['original_relative_path'].apply(
        lambda rel: os.path.join(DATASET_DIR, rel)
    )
    df['label'] = df['class_name'].map(LABEL_MAP)
    df['split'] = None
    print('Using unified labels.csv metadata for dataset provenance and image-level split.')
else:
    data = []
    for class_name, label in label_map.items():
        class_path = os.path.join(DATASET_DIR, class_name)
        if not os.path.exists(class_path):
            continue

        for filename in os.listdir(class_path):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                image_path = os.path.join(class_path, filename)
                data.append({
                    'image_path': image_path,
                    'label': label,
                    'class_name': class_name,
                    'source_dataset': 'UnifiedDataset',
                    'image_id': os.path.splitext(filename)[0],
                    'original_filename': filename,
                    'original_relative_path': os.path.relpath(image_path, DATASET_DIR),
                    'split': None
                })

    df = pd.DataFrame(data)
    print('Dataset labels.csv was not found; using directory-based provenance metadata.')

print('Total images:', len(df))
print(df.head())
print(df['class_name'].value_counts())
print('Source metadata columns available:', [col for col in ['source_dataset', 'image_id', 'original_filename', 'original_relative_path', 'split'] if col in df.columns])
print('No synthetic subject identifier is used because the unified dataset does not contain a reliable cross-source subject key.')


## 08. Train/Validation/Test Split


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='class_name', order=['Normal', 'Pneumonia', 'Tuberculosis'])
plt.title('Class Distribution')
plt.xlabel('Disease Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 09. Save Dataset Splits


In [ ]:
# Stratified image-level split: 70/15/15
# The unified dataset does not provide a reliable shared subject identifier across sources,
# so the split is performed at the image level while preserving class proportions.
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=SEED
)

train_df['split'] = 'train'
val_df['split'] = 'validation'
test_df['split'] = 'test'

print('Image-level stratified split used (70% train / 15% validation / 15% test).')
print('Train size:', len(train_df))
print('Validation size:', len(val_df))
print('Test size:', len(test_df))
print('Train class counts:\n', train_df['class_name'].value_counts())
print('Validation class counts:\n', val_df['class_name'].value_counts())
print('Test class counts:\n', test_df['class_name'].value_counts())
print('No synthetic subject identifier was created; the split is image-level and source-aware.')

## 10. Image Preprocessing


In [ ]:
train_df.to_csv(os.path.join(SPLIT_DIR, 'train.csv'), index=False)
val_df.to_csv(os.path.join(SPLIT_DIR, 'val.csv'), index=False)
test_df.to_csv(os.path.join(SPLIT_DIR, 'test.csv'), index=False)

print('Saved splits to:', SPLIT_DIR)

## 11. Data Augmentation

Data augmentation is applied only to the training set to improve generalization and reduce overfitting. For chest X-ray images, transformations should be moderate and medically realistic. This notebook uses random horizontal flipping and slight rotation to simulate variation without distorting clinically important patterns.

The validation and test sets remain unchanged so the evaluation reflects real unseen data quality.

In [ ]:
# 11. Data Augmentation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('Training augmentation configured.')

## 12. DataLoader

The dataset is converted into PyTorch DataLoader objects for efficient batch processing during training and evaluation. DataLoader settings such as batch size and shuffle behavior are important for stable optimization and reliable validation.

This notebook loads the training set with shuffling enabled and keeps the validation and test sets in evaluation order so metrics are consistent.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

print('Preprocessing configured for ImageNet normalization.')

## 13. Class Imbalance Handling


In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = Image.open(row['image_path']).convert('RGB')
        label = int(row['label'])

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset = ChestXrayDataset(train_df, train_transform)
val_dataset = ChestXrayDataset(val_df, val_test_transform)
test_dataset = ChestXrayDataset(test_df, val_test_transform)

print('Train dataset size:', len(train_dataset))
print('Validation dataset size:', len(val_dataset))
print('Test dataset size:', len(test_dataset))

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print('DataLoaders created successfully.')

## 14. Model Architecture Setup

This section prepares the four deep-learning models used in the thesis: ResNet-50, DenseNet-121, EfficientNetV2-S, and ConvNeXt-Tiny. All models are initialized with ImageNet pretrained weights and adapted for the 3-class lung disease problem.

The same training split, validation split, preprocessing, loss function, and evaluation metrics are used for all models so that their comparison is fair and scientifically meaningful.

In [ ]:
# 14. Model Architecture Setup
MODEL_BUILDERS = {
    'ResNet-50': models.resnet50,
    'DenseNet-121': models.densenet121,
    'EfficientNetV2-S': models.efficientnet_v2_s,
    'ConvNeXt-Tiny': models.convnext_tiny,
}

print('Model architectures ready:')
for name in MODEL_BUILDERS:
    print('-', name)

## 14.1 ResNet-50

ResNet-50 is a classic residual network that is widely used in medical image classification because it trains efficiently and preserves important feature information through skip connections. Its residual blocks help reduce vanishing-gradient issues and make deeper networks more stable.

In this thesis, the final fully connected layer is replaced with a 3-class classifier for Normal, Pneumonia, and Tuberculosis.

In [ ]:
class_counts = train_df['label'].value_counts().sort_index()
weights = len(train_df) / (len(class_counts) * class_counts)
class_weights = torch.tensor(weights.values, dtype=torch.float32).to(device)
class_weight_dict = {label: float(weight) for label, weight in zip(class_counts.index.tolist(), weights.values.tolist())}

print('Class counts:', class_counts.to_dict())
print('Class weights:', class_weight_dict)

criterion = nn.CrossEntropyLoss(weight=class_weights)

## 14.2 DenseNet-121

DenseNet-121 is a densely connected convolutional network that encourages feature reuse and efficient gradient flow. It is often useful in medical imaging where subtle patterns matter and where rich feature propagation can improve diagnosis.

In this thesis, the DenseNet-121 classifier head is replaced with a 3-class output layer for Normal, Pneumonia, and Tuberculosis.

In [ ]:
# 14.2 DenseNet-121 (active thesis model)
# This model is included in the final comparison set and is not left as an unused prototype.
densenet_model = build_densenet121(num_classes=3).to(device)
print('DenseNet-121 built successfully and included in the thesis comparison set.')
print('DenseNet-121 parameter summary:', sum(p.numel() for p in densenet_model.parameters()))

## 14.3 EfficientNetV2-S


In [ ]:
def build_resnet50(num_classes=3):
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

def build_densenet121(num_classes=3):
    weights = models.DenseNet121_Weights.DEFAULT
    model = models.densenet121(weights=weights)
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, num_classes)
    return model

def build_efficientnetv2s(num_classes=3):
    weights = models.EfficientNet_V2_S_Weights.DEFAULT
    model = models.efficientnet_v2_s(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def build_convnext_tiny(num_classes=3):
    weights = models.ConvNeXt_Tiny_Weights.DEFAULT
    model = models.convnext_tiny(weights=weights)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)
    return model

print('Model builders ready.')

## 14.4 ConvNeXt-Tiny


In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler=None, epochs=20, patience=5, min_delta=1e-4):
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': [],
        'val_f1': [],
        'lr': []
    }

    best_val_f1 = 0.0
    best_state = None
    best_epoch = 0
    stale_epochs = 0
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        val_true = []
        val_pred = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs, dim=1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                val_true.extend(labels.cpu().tolist())
                val_pred.extend(predicted.cpu().tolist())

        val_loss = val_loss / len(val_loader)
        val_acc = val_correct / val_total
        val_f1 = f1_score(val_true, val_pred, average='macro', zero_division=0)

        if scheduler is not None:
            scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['lr'].append(current_lr)

        print(f'Epoch {epoch + 1}/{epochs} | LR: {current_lr:.6f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f} | Val Loss: {val_loss:.4f}')

        if val_f1 > best_val_f1 + min_delta:
            best_val_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            stale_epochs = 0
        else:
            stale_epochs += 1

        if stale_epochs >= patience:
            print(f'Early stopping triggered at epoch {epoch + 1} due to no validation macro-F1 improvement for {patience} epochs.')
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    history['best_epoch'] = best_epoch + 1
    history['stopped_early'] = stale_epochs >= patience
    history['best_val_f1'] = best_val_f1
    history['training_time_sec'] = round(time.time() - start_time, 2)

    return history


## 15. Training Functions


In [ ]:
def freeze_backbone(model):
    for name, param in model.named_parameters():
        if not name.startswith('fc') and not name.startswith('classifier'):
            param.requires_grad = False
    return model


def unfreeze_deep_layers(model):
    for name, param in model.named_parameters():
        if ('layer4' in name or 'denseblock4' in name or 'features' in name or 'classifier' in name):
            param.requires_grad = True
    return model


def setup_model(model_name):
    if model_name == 'resnet50':
        model = build_resnet50(num_classes=3).to(device)
    elif model_name == 'densenet121':
        model = build_densenet121(num_classes=3).to(device)
    elif model_name == 'efficientnetv2s':
        model = build_efficientnetv2s(num_classes=3).to(device)
    elif model_name == 'convnexttiny':
        model = build_convnext_tiny(num_classes=3).to(device)
    else:
        raise ValueError(f'Unknown model: {model_name}')

    model = freeze_backbone(model)
    return model


def train_two_stage_model(model, train_loader, val_loader, criterion, epochs_stage1=8, epochs_stage2=12, patience=5):
    # Stage 1: train the classifier with the pretrained backbone frozen.
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs_stage1)

    print('Stage 1: train only the classifier head with the backbone frozen.')
    history_stage1 = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        epochs=epochs_stage1,
        patience=patience
    )

    # Stage 2: unfreeze deeper layers and fine-tune with a smaller learning rate.
    model = unfreeze_deep_layers(model)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=1e-5, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs_stage2)

    print('Stage 2: fine-tune the deeper layers with a smaller learning rate.')
    history_stage2 = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        epochs=epochs_stage2,
        patience=patience
    )

    history = {
        'train_loss': history_stage1['train_loss'] + history_stage2['train_loss'],
        'val_loss': history_stage1['val_loss'] + history_stage2['val_loss'],
        'train_acc': history_stage1['train_acc'] + history_stage2['train_acc'],
        'val_acc': history_stage1['val_acc'] + history_stage2['val_acc'],
        'val_f1': history_stage1['val_f1'] + history_stage2['val_f1'],
        'lr': history_stage1['lr'] + history_stage2['lr'],
        'best_epoch': history_stage2['best_epoch'],
        'best_val_f1': max(history_stage1['best_val_f1'], history_stage2['best_val_f1']),
        'training_time_sec': history_stage1['training_time_sec'] + history_stage2['training_time_sec'],
        'stopped_early': history_stage1['stopped_early'] or history_stage2['stopped_early']
    }
    return history


model_names = ['resnet50', 'densenet121', 'efficientnetv2s', 'convnexttiny']
results = {}

In [ ]:
for model_name in model_names:
    print(f'\n===== Training {model_name} =====')
    model = setup_model(model_name)
    history = train_two_stage_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        epochs_stage1=8,
        epochs_stage2=12,
        patience=5
    )
    results[model_name] = {
        'model': model,
        'history': history
    }

    torch.save(model.state_dict(), os.path.join(MODEL_DIR, f'{model_name}_best.pth'))
    print(f'{model_name} training finished. Best epoch metadata:', history['best_epoch'])

## 16. Model Training


In [ ]:
for model_name, value in results.items():
    history = value['history']

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title(f'{model_name} Accuracy Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title(f'{model_name} Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

## 17. Training Curves


In [ ]:
def evaluate_model(model, loader, class_names):
    model.eval()
    y_true = []
    y_pred = []
    y_prob = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            pred_labels = torch.argmax(outputs, dim=1).cpu().numpy()

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(pred_labels)
            y_prob.extend(probs)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    y_onehot = np.eye(len(class_names))[y_true]
    roc_auc = roc_auc_score(y_onehot, y_prob, multi_class='ovr', average='macro')

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )

    cm = confusion_matrix(y_true, y_pred)

    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'classification_report': report,
        'confusion_matrix': cm
    }

    return metrics, y_true, y_pred, y_prob

In [ ]:
def plot_confusion_matrix(cm, class_names, title='Confusion Matrix'):
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def get_model_complexity(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    model_size_mb = (param_bytes + buffer_bytes) / (1024 ** 2)
    return {
        'total_parameters': total_params,
        'trainable_parameters': trainable_params,
        'model_size_mb': model_size_mb
    }


def benchmark_inference_time(model, loader, device, repeats=20):
    model.eval()
    images, _ = next(iter(loader))
    images = images.to(device)
    timings = []

    with torch.no_grad():
        for _ in range(repeats):
            start = time.perf_counter()
            _ = model(images)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            elapsed_ms = (time.perf_counter() - start) * 1000 / images.size(0)
            timings.append(elapsed_ms)

    return float(np.mean(timings))


def plot_multiclass_roc(y_true, y_prob, class_names, title='Multiclass ROC Curves'):
    y_true_onehot = np.eye(len(class_names))[y_true]
    fig, ax = plt.subplots(figsize=(8, 6))

    all_fpr = np.unique(np.concatenate([
        roc_curve(y_true_onehot[:, i], y_prob[:, i])[0] for i in range(len(class_names))
    ]))

    for i, name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})', linewidth=2)

    tpr_macro = []
    for threshold in all_fpr:
        class_scores = []
        for i in range(len(class_names)):
            fpr_i, tpr_i, _ = roc_curve(y_true_onehot[:, i], y_prob[:, i])
            class_scores.append(np.interp(threshold, fpr_i, tpr_i))
        tpr_macro.append(np.mean(class_scores))

    macro_auc = auc(all_fpr, tpr_macro)
    ax.plot(all_fpr, tpr_macro, label=f'Macro-average (AUC={macro_auc:.3f})', linestyle='--', linewidth=2.5, color='black')

    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(title)
    ax.legend(loc='lower right')
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

## 18. Test Evaluation

After training, the best model checkpoint is evaluated on the held-out test set. This step measures how well the model generalizes to unseen X-ray images and confirms whether the training process resulted in a robust classifier.

The evaluation includes overall accuracy as well as macro precision, macro recall, macro F1-score, and ROC-AUC, which are all more informative than accuracy alone in a multiclass medical diagnosis task.

In [ ]:
# 18. Validation-Based Model Selection and Final Test Evaluation
# IMPORTANT: Use validation performance to select the model and keep the test set unseen until the final evaluation.
val_model_results = {}

for model_name, value in results.items():
    model = value['model']
    metrics, y_true, y_pred, y_prob = evaluate_model(model, val_loader, classes)
    complexity = get_model_complexity(model)
    inference_time_ms = benchmark_inference_time(model, val_loader, device)
    val_model_results[model_name] = {
        'metrics': metrics,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'complexity': complexity,
        'inference_time_ms': inference_time_ms
    }

    print(f'\n=== VALIDATION RESULTS: {model_name.upper()} ===')
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"Total Parameters: {complexity['total_parameters']:,}")
    print(f"Trainable Parameters: {complexity['trainable_parameters']:,}")
    print(f"Model Size (MB): {complexity['model_size_mb']:.2f}")
    print(f"Inference Time (ms/image): {inference_time_ms:.2f}")
    print(f"Training Time (s): {results[model_name]['history']['training_time_sec']:.2f}")
    print(metrics['classification_report'])
    plot_confusion_matrix(metrics['confusion_matrix'], classes, title=f'{model_name} Validation Confusion Matrix')
    plot_multiclass_roc(y_true, y_prob, classes, title=f'{model_name} Validation ROC Curves')

# Validation Macro-F1 selects the best architecture.
# The chosen architecture is then locked before any final test evaluation.
best_model_name = max(val_model_results, key=lambda name: val_model_results[name]['metrics']['f1'])
best_model = results[best_model_name]['model'].to(device)
final_model_name = best_model_name
final_model = best_model
model_eval_results = val_model_results

# Final test evaluation happens only once after the architecture has been selected.
final_test_metrics, y_true_test, y_pred_test, y_prob_test = evaluate_model(final_model, test_loader, classes)

print(f'\nBEST ARCHITECTURE SELECTED USING VALIDATION MACRO-F1: {final_model_name}')
print(f'Architecture locked for thesis pipeline: {final_model_name}')
print(final_test_metrics['classification_report'])
print(f'Final test ROC-AUC: {final_test_metrics["roc_auc"]:.4f}')
print(f'Final test Accuracy: {final_test_metrics["accuracy"]:.4f}')
print(f'Final test F1: {final_test_metrics["f1"]:.4f}')


## 19. Confusion Matrices


In [ ]:
# Validation confusion matrices are used for model selection.
# The final test confusion matrix is generated only once for the selected model.
for model_name, result in model_eval_results.items():
    metrics = result['metrics']
    print(f'\n=== {model_name.upper()} VALIDATION CONFUSION MATRIX ===')
    plot_confusion_matrix(metrics['confusion_matrix'], classes, title=f'{model_name} Validation Confusion Matrix')

plot_confusion_matrix(final_test_metrics['confusion_matrix'], classes, title=f'{best_model_name} Final Test Confusion Matrix')

## 20. Classification Reports

The classification report provides per-class metrics for each disease category. It includes precision, recall, F1-score, and support for Normal, Pneumonia, and Tuberculosis.

This is especially important in medical AI because a model may show good overall accuracy while performing poorly on a smaller class such as Tuberculosis.

In [ ]:
# 20. Final Test Classification Report
# Report the final held-out test performance of the validation-selected best model.
report = classification_report(y_true_test, y_pred_test, target_names=classes, digits=4)
print(report)

result_path = os.path.join(RESULT_DIR, f'{best_model_name}_classification_report.txt')
with open(result_path, 'w', encoding='utf-8') as f:
    f.write(report)
print('Saved final classification report to:', result_path)


## 21. ROC-AUC

ROC-AUC is used to assess the ranking ability of the model across all classes. Because the task is multiclass classification, the implementation uses a one-vs-rest (OvR) strategy and computes the macro ROC-AUC across all three classes.

This metric complements accuracy and F1-score by showing how strongly the model separates disease classes based on predicted probabilities.

In [ ]:
# 21. Final Test ROC-AUC
# Compute multiclass ROC-AUC on the final held-out test set for the selected model.
if len(np.unique(y_true_test)) > 1:
    y_onehot = np.eye(len(classes))[y_true_test]
    roc_auc = roc_auc_score(y_onehot, y_prob_test, multi_class='ovr', average='macro')
    print(f'Macro ROC-AUC: {roc_auc:.4f}')
    plot_multiclass_roc(y_true_test, y_prob_test, classes, title=f'{best_model_name} Final Test ROC Curves')
else:
    print('Not enough classes in test set to compute ROC-AUC.')


## 22. Model Comparison

This section compares all four architectures using the same dataset split and evaluation metrics. The comparison includes accuracy, precision, recall, F1-score, and ROC-AUC.

The final model should be selected based on both predictive performance and clinical usefulness. For a medical dataset, macro F1 and class-wise recall are often more important than overall accuracy alone.

In [ ]:
# 22. Model Comparison
# Comparison is based on validation performance to avoid test-set leakage.
comparison_rows = []
for name, result in model_eval_results.items():
    metrics = result['metrics']
    complexity = result['complexity']
    comparison_rows.append({
        'Model': name,
        'Total Parameters': complexity['total_parameters'],
        'Trainable Parameters': complexity['trainable_parameters'],
        'Model Size (MB)': round(complexity['model_size_mb'], 2),
        'Training Time (s)': round(results[name]['history']['training_time_sec'], 2),
        'Inference Time (ms/image)': round(result['inference_time_ms'], 3),
        'Validation Accuracy': round(metrics['accuracy'], 4),
        'Validation Precision': round(metrics['precision'], 4),
        'Validation Recall': round(metrics['recall'], 4),
        'Validation F1': round(metrics['f1'], 4),
        'Validation ROC-AUC': round(metrics['roc_auc'], 4),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(by='Validation F1', ascending=False).reset_index(drop=True)
comparison_path = os.path.join(RESULT_DIR, 'model_comparison.csv')
comparison_df.to_csv(comparison_path, index=False)

print(comparison_df)
print('Validation-based comparison table saved to:', comparison_path)

In [ ]:
summary_rows = []
for model_name, result in model_eval_results.items():
    m = result['metrics']
    summary_rows.append({
        'Model': model_name,
        'Accuracy': round(m['accuracy'], 4),
        'Precision': round(m['precision'], 4),
        'Recall': round(m['recall'], 4),
        'F1': round(m['f1'], 4),
        'ROC-AUC': round(m['roc_auc'], 4)
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df)

## 23. Grad-CAM

Grad-CAM is used to explain which regions of an X-ray image influenced the model's prediction. This is important in medical AI because it helps verify whether the model is focusing on clinically relevant anatomy rather than irrelevant background noise.

For each class, the notebook now selects several correctly classified representative images and visualizes the original X-ray alongside its Grad-CAM overlay. This provides a systematic XAI analysis aligned with the thesis objective of explainable AI.

In [ ]:
# 23. Grad-CAM
# The best architecture was already locked after validation-based selection.
best_model_name = final_model_name
best_model = final_model.to(device)
print('Locked final model for XAI analysis:', best_model_name)

## 24. XAI Comparison

This section compares the explainability of the four architectures on the same representative medical cases. For each disease class, the notebook now includes both correctly classified X-rays and misclassified X-rays so the thesis can assess whether each model focuses on clinically relevant lung regions consistently and also understand failure patterns. The misclassified examples are particularly useful for academic discussion of model limitation and error analysis.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hooks = []
        self._register_hooks()

    def _register_hooks(self):
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        def forward_hook(module, input, output):
            self.activations = output.detach()

        self.hooks.append(self.target_layer.register_forward_hook(forward_hook))
        self.hooks.append(self.target_layer.register_full_backward_hook(backward_hook))

    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()

    def __call__(self, input_tensor, class_index=None):
        self.model.zero_grad()
        logits = self.model(input_tensor)

        if class_index is None:
            class_index = torch.argmax(logits, dim=1).item()

        score = logits[:, class_index]
        score.backward(retain_graph=True)

        gradients = self.gradients
        activations = self.activations

        weights = torch.mean(gradients, dim=(2, 3), keepdim=True)
        heatmap = torch.sum(weights * activations, dim=1, keepdim=True)
        heatmap = torch.relu(heatmap)
        heatmap = F.interpolate(heatmap, size=(input_tensor.shape[-2], input_tensor.shape[-1]), mode='bilinear', align_corners=False)
        heatmap = heatmap[0, 0].cpu().detach().numpy()
        heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
        return heatmap


def get_gradcam_target_layer(model):
    model_name = type(model).__name__.lower()

    # The target layer must be verified against the exact torchvision architecture being used.
    # For Colab-based torchvision builds, the final convolutional layer in the feature extractor is the
    # most stable Grad-CAM target for these four CNN families.
    if 'resnet' in model_name:
        conv_layers = [m for m in model.layer4.modules() if isinstance(m, nn.Conv2d)]
        target_layer = conv_layers[-1]
    elif 'densenet' in model_name:
        conv_layers = [m for m in model.features.denseblock4.modules() if isinstance(m, nn.Conv2d)]
        target_layer = conv_layers[-1]
    elif 'efficientnet' in model_name:
        conv_layers = [m for m in model.features.modules() if isinstance(m, nn.Conv2d)]
        target_layer = conv_layers[-1]
    elif 'convnext' in model_name:
        conv_layers = [m for m in model.features.modules() if isinstance(m, nn.Conv2d)]
        target_layer = conv_layers[-1]
    else:
        raise ValueError(f'Unsupported architecture for Grad-CAM: {model_name}')

    print(f'Grad-CAM target verified for {model_name}: {target_layer}')
    return target_layer


def generate_gradcam(model, image_tensor, class_index=None, target_layer=None):
    if target_layer is None:
        target_layer = get_gradcam_target_layer(model)

    gradcam = GradCAM(model, target_layer)
    heatmap = gradcam(image_tensor.unsqueeze(0).to(device), class_index=class_index)
    gradcam.remove_hooks()
    return heatmap


def verify_gradcam_for_architecture(model, image_tensor, class_index=0):
    target_layer = get_gradcam_target_layer(model)
    heatmap = generate_gradcam(model, image_tensor, class_index=class_index, target_layer=target_layer)

    img_np = image_tensor.cpu().permute(1, 2, 0).numpy()
    img_np = (img_np * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np, 0, 1)

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img_np)
    plt.title(f'{type(model).__name__} Original')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(img_np)
    plt.imshow(heatmap, cmap='jet', alpha=0.35)
    plt.title(f'{type(model).__name__} Grad-CAM')
    plt.axis('off')

    plt.tight_layout()
    plt.show()
    return target_layer


def get_representative_examples(model, dataset, class_id, mode='correct', max_examples=1):
    selected = []
    model.eval()

    for idx, (image, label) in enumerate(dataset):
        if int(label) != class_id:
            continue

        with torch.no_grad():
            logits = model(image.unsqueeze(0).to(device))
            probs = torch.softmax(logits, dim=1)
            pred_idx = int(torch.argmax(logits, dim=1).item())
            confidence = float(probs[0, pred_idx].item())

        if mode == 'correct' and pred_idx == class_id:
            selected.append((idx, image, label, pred_idx, confidence))
        elif mode == 'misclassified' and pred_idx != class_id:
            selected.append((idx, image, label, pred_idx, confidence))

        if len(selected) >= max_examples:
            break

    return selected


def plot_model_xai_comparison(model_bank, dataset, class_id, class_name, mode='correct', max_examples=1):
    examples = []
    for model_name, model in model_bank.items():
        matched = get_representative_examples(model, dataset, class_id, mode=mode, max_examples=max_examples)
        if matched:
            examples.append((model_name, matched[0]))

    if not examples:
        if mode == 'correct':
            print(f'No correctly classified examples found for class: {class_name}')
        else:
            print(f'No misclassified examples found for class: {class_name}')
        return

    fig, axes = plt.subplots(len(examples), 2, figsize=(10, 4 * len(examples)))
    if len(examples) == 1:
        axes = np.array([axes], dtype=object)

    for row_idx, (model_name, (idx, image, true_label, pred_idx, confidence)) in enumerate(examples):
        img_np = image.cpu().permute(1, 2, 0).numpy()
        img_np = (img_np * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np, 0, 1)

        model = model_bank[model_name]
        with torch.no_grad():
            logits = model(image.unsqueeze(0).to(device))
            pred_idx = int(torch.argmax(logits, dim=1).item())
            confidence = float(torch.softmax(logits, dim=1)[0, pred_idx].item())

        heatmap = generate_gradcam(model, image.cpu(), class_index=pred_idx)

        axes[row_idx, 0].imshow(img_np)
        if mode == 'correct':
            axes[row_idx, 0].set_title(f'{model_name}\nOriginal X-ray\nTrue: {classes[int(true_label)]}\nPred: {classes[pred_idx]}')
        else:
            axes[row_idx, 0].set_title(f'{model_name}\nOriginal X-ray\nTrue: {classes[int(true_label)]}\nPredicted: {classes[pred_idx]}\nConfidence: {confidence:.2f}')
        axes[row_idx, 0].axis('off')

        axes[row_idx, 1].imshow(img_np)
        axes[row_idx, 1].imshow(heatmap, cmap='jet', alpha=0.35)
        if mode == 'correct':
            axes[row_idx, 1].set_title(f'{model_name}\nGrad-CAM Overlay')
        else:
            axes[row_idx, 1].set_title(f'{model_name}\nMisclassification Grad-CAM\nModel focused on: {classes[pred_idx]}')
        axes[row_idx, 1].axis('off')

    if mode == 'correct':
        fig.suptitle(f'Cross-Model Grad-CAM Comparison for {class_name} (Correct Predictions)', fontsize=14)
    else:
        fig.suptitle(f'Cross-Model Grad-CAM Comparison for {class_name} (Misclassified Cases)', fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


xai_model_bank = {name: value['model'].to(device) for name, value in results.items()}

print('\n=== GRAD-CAM TARGET LAYER VERIFICATION ===')
for model_name, model in xai_model_bank.items():
    sample_image, sample_label = test_dataset[0]
    print(f'Visual verification for {model_name}...')
    verify_gradcam_for_architecture(model, sample_image, class_index=int(sample_label))

for class_id, class_name in enumerate(classes):
    print(f'\n=== CROSS-MODEL XAI COMPARISON: {class_name} (Correct Predictions) ===')
    plot_model_xai_comparison(xai_model_bank, test_dataset, class_id, class_name, mode='correct', max_examples=1)

    print(f'\n=== CROSS-MODEL XAI COMPARISON: {class_name} (Misclassified Cases) ===')
    plot_model_xai_comparison(xai_model_bank, test_dataset, class_id, class_name, mode='misclassified', max_examples=1)

## 25. Best Model Selection


In [ ]:
# Example Grad-CAM visualization for a single image
sample_index = 0
sample_image, sample_label = test_dataset[sample_index]
sample_image = sample_image.unsqueeze(0).to(device)
sample_pred = best_model(sample_image)
pred_class = int(torch.argmax(sample_pred, dim=1).item())

heatmap = generate_gradcam(
    best_model,
    sample_image.squeeze(0).cpu(),
    class_index=pred_class
)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
img_np = sample_image.squeeze(0).cpu().permute(1, 2, 0).numpy()
img_np = (img_np * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
img_np = np.clip(img_np, 0, 1)
plt.imshow(img_np)
plt.title(f'Original X-ray | True: {classes[sample_label]} | Pred: {classes[pred_class]}')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_np)
plt.imshow(heatmap, cmap='jet', alpha=0.35)
plt.title('Grad-CAM Heatmap Overlay')
plt.axis('off')

plt.tight_layout()
plt.show()

## 26. Save Final Model

Once the best model is identified, its weights are saved to the Models directory. This allows the final trained model to be reused for inference, evaluation, or future comparison without retraining.

The saved checkpoint also supports reproducibility and serves as the final model artifact for the thesis report.

In [ ]:
# 26. Save Final Model
# Save a richer checkpoint to support reproducibility and future inference.
# This includes model weights, class mapping, image size, normalization, configuration, and training metadata.

best_model_path = os.path.join(MODEL_DIR, f'{best_model_name}_best_checkpoint.pth')
experiment_config_path = os.path.join(RESULT_DIR, 'experiment_config.json')

checkpoint = {
    'model_name': best_model_name,
    'model_state_dict': best_model.state_dict(),
    'class_to_idx': {name: idx for idx, name in enumerate(classes)},
    'classes': classes,
    'image_size': IMG_SIZE,
    'normalization_mean': [0.485, 0.456, 0.406],
    'normalization_std': [0.229, 0.224, 0.225],
    'training_config': CONFIG,
    'best_validation_f1': float(val_model_results[best_model_name]['metrics']['f1']),
    'best_validation_accuracy': float(val_model_results[best_model_name]['metrics']['accuracy']),
    'best_validation_epoch': int(results[best_model_name]['history'].get('best_epoch', 0)),
    'optimizer_state_dict': None,
    'scheduler_state_dict': None,
}

if 'optimizer' in locals():
    checkpoint['optimizer_state_dict'] = optimizer.state_dict()
if 'scheduler' in locals():
    checkpoint['scheduler_state_dict'] = scheduler.state_dict()

with open(experiment_config_path, 'w', encoding='utf-8') as f:
    json.dump(CONFIG, f, indent=2)

torch.save(checkpoint, best_model_path)

print('Final selected model for thesis evaluation:', best_model_name)
print('Experiment config saved at:', experiment_config_path)
print('Best model checkpoint saved at:', best_model_path)

## 27. Final Results


In [ ]:
# Save summary table
summary_df.to_csv(os.path.join(RESULT_DIR, 'model_comparison.csv'), index=False)

# Save final selected-model predictions.
best_model_predictions = []
for idx in range(len(test_dataset)):
    image, true_label = test_dataset[idx]
    image_tensor = image.unsqueeze(0).to(device)
    with torch.no_grad():
        prob = torch.softmax(best_model(image_tensor), dim=1).cpu().numpy()[0]
    pred_idx = int(np.argmax(prob))
    best_model_predictions.append({
        'true_label': int(true_label),
        'true_class': classes[int(true_label)],
        'predicted_label': pred_idx,
        'predicted_class': classes[pred_idx],
        'prob_normal': float(prob[0]),
        'prob_pneumonia': float(prob[1]),
        'prob_tuberculosis': float(prob[2]),
    })

predictions_df = pd.DataFrame(best_model_predictions)
predictions_df.to_csv(os.path.join(RESULT_DIR, 'best_model_predictions.csv'), index=False)
summary_df.to_csv(os.path.join(RESULT_DIR, 'summary_results.csv'), index=False)

print('Results saved in:', RESULT_DIR)
print('Best model checkpoint path:', os.path.join(MODEL_DIR, f'{best_model_name}_best_checkpoint.pth'))

## 28. Export Results for Thesis


In [ ]:
# Export thesis results
# Final test metrics are generated from the selected validation-best model only.
summary_df.to_csv(os.path.join(RESULT_DIR, 'model_comparison.csv'), index=False)

experiment_config_path = os.path.join(RESULT_DIR, 'experiment_config.json')
with open(experiment_config_path, 'w', encoding='utf-8') as f:
    json.dump(CONFIG, f, indent=2)

best_model_predictions = []
for idx in range(len(test_dataset)):
    image, true_label = test_dataset[idx]
    image_tensor = image.unsqueeze(0).to(device)
    with torch.no_grad():
        prob = torch.softmax(best_model(image_tensor), dim=1).cpu().numpy()[0]
    pred_idx = int(np.argmax(prob))
    best_model_predictions.append({
        'true_label': int(true_label),
        'true_class': classes[int(true_label)],
        'predicted_label': pred_idx,
        'predicted_class': classes[pred_idx],
        'prob_normal': float(prob[0]),
        'prob_pneumonia': float(prob[1]),
        'prob_tuberculosis': float(prob[2]),
    })

predictions_df = pd.DataFrame(best_model_predictions)
predictions_df.to_csv(os.path.join(RESULT_DIR, 'best_model_predictions.csv'), index=False)
summary_df.to_csv(os.path.join(RESULT_DIR, 'summary_results.csv'), index=False)

# Explicitly generate the final best-model learning curves from the stored history.
history = results[best_model_name]['history']

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.title(f'{best_model_name} Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'best_model_accuracy_curve.png'), dpi=200)
plt.close()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title(f'{best_model_name} Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'best_model_loss_curve.png'), dpi=200)
plt.close()

print('Saved experiment config to:', experiment_config_path)
print('Saved final results to:', RESULT_DIR)
print('Saved best-model predictions to:', os.path.join(RESULT_DIR, 'best_model_predictions.csv'))
print('Saved final accuracy curve to:', os.path.join(RESULT_DIR, 'best_model_accuracy_curve.png'))
print('Saved final loss curve to:', os.path.join(RESULT_DIR, 'best_model_loss_curve.png'))
print('Final selected model for thesis testing:', best_model_name)